# encoder-decoder-symmetric — worked example 3: Stride Product Verification: Diagnosing Asymmetric Configs

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `encoder-decoder-symmetric`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The encoder-decoder symmetry condition is simple: the product of all encoder downsampling factors must equal the product of all decoder upsampling factors. If these products differ, or if the input spatial size is not divisible by the encoder's total stride, the output will not match the input shape. Checking these two conditions before building the model prevents difficult-to-debug shape errors during the forward pass.

## Worked solution

We write a function that takes encoder strides and decoder upsamples and checks the symmetry conditions mathematically — without building a module.

**Step 1 — compute products:** Multiply all strides together for `enc_product` and all upsample factors for `dec_product`.

**Step 2 — test product match:** If `enc_product != dec_product`, the config is asymmetric and the output will be the wrong spatial size.

**Step 3 — test input divisibility:** If `input_hw % enc_product != 0`, the pooling stages will use floor-division arithmetic, producing a spatial size that doesn't upsample back to the original.

**Step 4 — predict output:** `predicted_hw = (input_hw // enc_product) * dec_product`. Only when both conditions are satisfied does `predicted_hw == input_hw`.

We test three configs: one valid, one with a product mismatch, and one where the input is not divisible by the stride product.

In [ ]:
import torch as t

def check_encoder_decoder_symmetry(encoder_strides, decoder_upsamples, input_hw):
    enc_prod = 1
    for s in encoder_strides:
        enc_prod *= s
    dec_prod = 1
    for u in decoder_upsamples:
        dec_prod *= u

    product_ok = (enc_prod == dec_prod)
    divisible  = (input_hw % enc_prod == 0)
    predicted  = (input_hw // enc_prod) * dec_prod
    symmetric  = product_ok and divisible
    return {
        'enc_product': enc_prod,
        'dec_product': dec_prod,
        'product_ok': product_ok,
        'input_divisible': divisible,
        'predicted_output_hw': predicted,
        'symmetric': symmetric,
    }

# Case 1: perfectly symmetric
r1 = check_encoder_decoder_symmetry([2, 2, 2], [2, 2, 2], 32)
print("Case 1 (3 pools, 3 upsamples, H=32):", r1)
assert r1['symmetric'] and r1['predicted_output_hw'] == 32

# Case 2: product mismatch (3 downsamples, 2 upsamples)
r2 = check_encoder_decoder_symmetry([2, 2, 2], [2, 2], 32)
print("Case 2 (product mismatch):", r2)
assert not r2['product_ok']
assert r2['predicted_output_hw'] != 32

# Case 3: not divisible (H=30 with stride product 8)
r3 = check_encoder_decoder_symmetry([2, 2, 2], [2, 2, 2], 30)
print("Case 3 (H=30, not divisible by 8):", r3)
assert not r3['input_divisible']
assert r3['predicted_output_hw'] != 30

print("All cases verified.")